In [ ]:
# -- coding: utf-8 --

from pathlib import Path
import numpy as np
import pandas as pd
import rasterio
from tqdm.auto import tqdm


# =========================
# USER SETTINGS
# =========================
SITE = "Timor_part1"
OUT_DIR = Path("../") / SITE
SCENES_DIR = Path("../") / SITE/ "landsat_c2_l2_extracted"
OUT_SST_DIR = Path("../") / SITE / "landsat_sst_outputs"
OUT_SST_DIR.mkdir(parents=True, exist_ok=True)

WRITE_GEOTIFF = True
OVERWRITE = False  # set True to re-generate even if output exists

NODATA_OUT = -9999.0

# Landsat C2 L2: ST_B10 -> Kelvin -> Celsius
ST_SCALE  = 0.00341802
ST_OFFSET = 149.0

# =========================
# QA_PIXEL bits (Landsat C2 L2)
# =========================
BIT_DILATED_CLOUD = 1
BIT_CIRRUS        = 2
BIT_CLOUD         = 3
BIT_CLOUD_SHADOW  = 4
# (bit 5 is snow; don't use it for cloud)

MASK_DILATED = True
MASK_CIRRUS  = True

# =========================
# OPTIONAL ST_QA filter
# =========================
# Start with False. If you still see warm bias from cloud edges/haze, turn on.
USE_ST_QA_FILTER = False
ST_QA_MAX_KEEP = 10  # if enabled, try 5–15 (NOT 1)

# =========================
# DEBUG / SUMMARY
# =========================
PRINT_EVERY = 25  # print one-line debug every N processed scenes
SUMMARY_CSV = OUT_SST_DIR / "sst_batch_summary.csv"


# =========================
# HELPERS
# =========================
def find_band_files(scene_dir: Path):
    """Return (st_b10_path, qa_pixel_path, st_qa_path or None)."""
    st_candidates   = list(scene_dir.glob("*_ST_B10.TIF")) + list(scene_dir.glob("*ST_B10.TIF"))
    qa_candidates   = list(scene_dir.glob("*_QA_PIXEL.TIF")) + list(scene_dir.glob("*QA_PIXEL.TIF"))
    stqa_candidates = list(scene_dir.glob("*_ST_QA.TIF")) + list(scene_dir.glob("*ST_QA.TIF"))

    st   = st_candidates[0] if st_candidates else None
    qa   = qa_candidates[0] if qa_candidates else None
    stqa = stqa_candidates[0] if stqa_candidates else None
    return st, qa, stqa


def qa_clear_mask(qa: np.ndarray) -> np.ndarray:
    """True = keep pixel (clear)."""
    dilated = (qa & (1 << BIT_DILATED_CLOUD)) != 0
    cirrus  = (qa & (1 << BIT_CIRRUS)) != 0
    cloud   = (qa & (1 << BIT_CLOUD)) != 0
    shadow  = (qa & (1 << BIT_CLOUD_SHADOW)) != 0

    bad = cloud | shadow
    if MASK_CIRRUS:
        bad |= cirrus
    if MASK_DILATED:
        bad |= dilated

    return ~bad


def compute_sst_c_from_st_b10(st_b10: np.ndarray) -> np.ndarray:
    """ST_B10 is a scaled surface temperature product (Kelvin). Convert to Celsius."""
    st_k = st_b10.astype(np.float32) * ST_SCALE + ST_OFFSET
    return (st_k - 273.15).astype(np.float32)


def finite_stats(arr: np.ndarray):
    x = arr[np.isfinite(arr)]
    if x.size == 0:
        return {"n": 0, "min": np.nan, "max": np.nan, "mean": np.nan}
    return {
        "n": int(x.size),
        "min": float(np.nanmin(x)),
        "max": float(np.nanmax(x)),
        "mean": float(np.nanmean(x)),
    }


def write_geotiff(out_path: Path, arr: np.ndarray, ref_profile: dict):
    profile = ref_profile.copy()
    profile.update(
        driver="GTiff",
        dtype="float32",
        count=1,
        nodata=float(NODATA_OUT),
        compress="deflate",
        tiled=True,
        BIGTIFF="IF_SAFER",
    )
    with rasterio.open(out_path, "w", **profile) as dst:
        dst.write(arr, 1)


# =========================
# MAIN (ALL SCENES)
# =========================
scene_dirs = sorted([p for p in SCENES_DIR.iterdir() if p.is_dir()])
print(f"Found {len(scene_dirs)} scene folders in: {SCENES_DIR}")

summary_rows = []
written = 0
skipped_exists = 0
skipped_missing = 0
errors = 0
processed = 0

for idx, scene_dir in enumerate(tqdm(scene_dirs, desc="Batch SST")):
    scene_name = scene_dir.name
    out_tif = OUT_SST_DIR / f"{scene_name}_SST_C.tif"

    st_path, qa_path, stqa_path = find_band_files(scene_dir)

    if st_path is None or qa_path is None:
        skipped_missing += 1
        summary_rows.append({
            "scene": scene_name,
            "status": "skip_missing_bands",
            "has_ST_B10": st_path is not None,
            "has_QA_PIXEL": qa_path is not None,
            "has_ST_QA": stqa_path is not None,
        })
        continue

    if (not OVERWRITE) and out_tif.exists() and out_tif.stat().st_size > 1000:
        skipped_exists += 1
        summary_rows.append({
            "scene": scene_name,
            "status": "skip_exists",
            "out_tif": str(out_tif),
        })
        continue

    try:
        with rasterio.open(st_path) as st_src, rasterio.open(qa_path) as qa_src:
            st = st_src.read(1)
            qa = qa_src.read(1)

            if st.shape != qa.shape:
                # unexpected for Landsat L2, but handle gracefully
                raise RuntimeError(f"Shape mismatch ST vs QA: {st.shape} vs {qa.shape}")

            # 1) QA-based clear mask
            keep = qa_clear_mask(qa)

            # 2) Mask ST nodata
            st_nodata = st_src.nodata
            if st_nodata is not None:
                keep &= (st != st_nodata)

            # 3) Optional ST_QA filter
            used_stqa = False
            if USE_ST_QA_FILTER and stqa_path is not None:
                with rasterio.open(stqa_path) as stqa_src:
                    stqa = stqa_src.read(1)
                if stqa.shape == st.shape:
                    keep &= (stqa <= ST_QA_MAX_KEEP)
                    used_stqa = True
                else:
                    # don't kill the scene just because ST_QA isn't aligned
                    summary_rows.append({
                        "scene": scene_name,
                        "status": "warn_stqa_shape_mismatch",
                        "st_shape": str(st.shape),
                        "stqa_shape": str(stqa.shape),
                    })

            # 4) Compute SST
            sst = compute_sst_c_from_st_b10(st)

            # 5) Apply mask to output
            sst_masked = np.where(keep, sst, NODATA_OUT).astype(np.float32)

            # Stats
            keep_pct = 100.0 * float(keep.mean()) if keep.size else 0.0

            sst_stats = finite_stats(np.where(st == st_nodata, np.nan, sst) if st_nodata is not None else sst)
            masked_stats = finite_stats(np.where(sst_masked == NODATA_OUT, np.nan, sst_masked))

            # Write
            if WRITE_GEOTIFF:
                write_geotiff(out_tif, sst_masked, st_src.profile)
                written += 1

            processed += 1
            summary_rows.append({
                "scene": scene_name,
                "status": "written",
                "keep_pct": keep_pct,
                "used_stqa": used_stqa,
                "sst_mean": sst_stats["mean"],
                "sst_min": sst_stats["min"],
                "sst_max": sst_stats["max"],
                "sst_masked_mean": masked_stats["mean"],
                "sst_masked_min": masked_stats["min"],
                "sst_masked_max": masked_stats["max"],
                "masked_n": masked_stats["n"],
                "out_tif": str(out_tif),
            })

            if PRINT_EVERY and (processed % PRINT_EVERY == 0):
                print(f"[DEBUG] {scene_name}: keep={keep_pct:.1f}% "
                      f"mean(before)={sst_stats['mean']:.2f} mean(after)={masked_stats['mean']:.2f} "
                      f"used_stqa={used_stqa}")

    except Exception as e:
        errors += 1
        summary_rows.append({
            "scene": scene_name,
            "status": "error",
            "error": repr(e),
            "st_path": str(st_path),
            "qa_path": str(qa_path),
            "stqa_path": str(stqa_path) if stqa_path else "",
        })

print("\n[DONE]")
print(f"written={written}, processed={processed}, skipped_exists={skipped_exists}, skipped_missing={skipped_missing}, errors={errors}")
df = pd.DataFrame(summary_rows)
df.to_csv(SUMMARY_CSV, index=False)
print(f"Wrote summary CSV: {SUMMARY_CSV}")

Found 232 scene folders in: ../Timor_part1/landsat_c2_l2_extracted


Batch SST:   0%|          | 0/232 [00:00<?, ?it/s]

[DEBUG] LC08_L2SP_110066_20210131_20210302_02_T2: keep=7.2% mean(before)=-102.30 mean(after)=-28.95 used_stqa=False
